In [ ]:
from seleniumbase import SB
from bs4 import BeautifulSoup
import pandas as pd

def extraer_tasas_sbs(html, fecha, moneda):
    
    soup = BeautifulSoup(html, "html.parser")


    # IDs según la moneda

    if moneda == "MN":

        tabla_id = (
            "ctl00_cphContent_rpgActualMn_"
            "ctl00_DataZone_DT"
        )

        tabla_creditos_id = (
            "ctl00_cphContent_rpgActualMn_OT"
        )

    elif moneda == "ME":

        tabla_id = (
            "ctl00_cphContent_rpgActualMex_"
            "ctl00_DataZone_DT"
        )

        tabla_creditos_id = (
            "ctl00_cphContent_rpgActualMex_OT"
        )

    else:
        raise ValueError(
            "La moneda debe ser 'MN' o 'ME'"
        )

    # --------------------------------------------------------
    # TABLA DE TASAS
    # --------------------------------------------------------

    tabla_datos = soup.find(
        "table",
        id=tabla_id
    )

    if tabla_datos is None:
        raise ValueError(
            "No se encontró la tabla de tasas."
        )

    filas = tabla_datos.find_all("tr")

    datos = []

    for fila in filas:

        celdas = fila.find_all(
            ["th", "td"]
        )

        valores = [
            celda.get_text(
                " ",
                strip=True
            )
            for celda in celdas
        ]

        datos.append(valores)

    # Primera fila = bancos
    bancos = datos[0]

    # Resto = tasas
    tasas = datos[1:]

    # --------------------------------------------------------
    # DATAFRAME
    # --------------------------------------------------------

    df = pd.DataFrame(
        tasas,
        columns=bancos
    )

    # --------------------------------------------------------
    # TABLA DE TIPOS DE CRÉDITO
    # --------------------------------------------------------

    tabla_creditos = soup.find(
        "table",
        id=tabla_creditos_id
    )

    if tabla_creditos is None:
        raise ValueError(
            "No se encontró la tabla de créditos."
        )

    tipos_credito = [
        td.get_text(
            " ",
            strip=True
        )
        for td in tabla_creditos.find_all(
            "td",
            class_=lambda x:
                x and "rpgRowHeader" in x
        )
    ]

    # VALIDACIÓN

    if len(df) != len(tipos_credito):

        raise ValueError(
            f"No coinciden las filas: "
            f"{len(df)} tasas vs "
            f"{len(tipos_credito)} créditos"
        )

    # AGREGAR TIPO DE CRÉDITO

    df.insert(
        0,
        "tipo_credito",
        tipos_credito
    )

    # IDENTIFICAR GRUPOS

    grupos_principales = [
        "Corporativos",
        "Grandes Empresas",
        "Medianas Empresas",
        "Pequeñas Empresas",
        "Microempresas",
        "Consumo",
        "Hipotecarios"
    ]

    grupo_actual = None
    grupos = []

    for credito in df["tipo_credito"]:

        if credito in grupos_principales:
            grupo_actual = credito

        grupos.append(grupo_actual)

    df.insert(
        1,
        "grupo_credito",
        grupos
    )

    # FECHA Y MONEDA

    df.insert(
        0,
        "fecha",
        fecha
    )

    df.insert(
        1,
        "moneda",
        moneda
    )
    # CONVERTIR TASAS A NUMÉRICO

    columnas_bancos = [
        col
        for col in df.columns
        if col not in [
            "fecha",
            "moneda",
            "tipo_credito",
            "grupo_credito"
        ]
    ]

    for banco in columnas_bancos:

        df[banco] = pd.to_numeric(
            df[banco],
            errors="coerce"
        )

    return df

# FUNCIÓN PRINCIPAL

def obtener_tasas_sbs(
    fecha,
    moneda="MN",
    grupo=None
):

    moneda = moneda.upper()

    if moneda not in ["MN", "ME"]:
        raise ValueError(
            "moneda debe ser 'MN' o 'ME'"
        )

    # --------------------------------------------------------
    # SELECTORES SEGÚN MONEDA
    # --------------------------------------------------------

    if moneda == "MN":
        selector_moneda = "#ctl00_cphContent_lbtnMn"

    else:
        selector_moneda = "#ctl00_cphContent_lbtnMex"

    # --------------------------------------------------------
    # SCRAPING
    # --------------------------------------------------------

    with SB(headless=False) as sb:

        # Abrir SBS
        sb.open(URL)

        # Esperar fecha
        sb.wait_for_element_visible(
            "#ctl00_cphContent_rdpDate_dateInput"
        )

        # Ingresar fecha
        sb.clear(
            "#ctl00_cphContent_rdpDate_dateInput"
        )

        sb.type(
            "#ctl00_cphContent_rdpDate_dateInput",
            fecha
        )

        # Consultar
        sb.click(
            "#ctl00_cphContent_btnConsultar"
        )

        # Esperar confirmación
        sb.wait_for_text(
            fecha,
            "#ctl00_cphContent_lblMensajeFecha",
            timeout=20
        )

        # Seleccionar moneda
        sb.click(
            selector_moneda
        )

        # Obtener HTML
        html = sb.get_page_source()

    # --------------------------------------------------------
    # EXTRAER INFORMACIÓN
    # --------------------------------------------------------

    df = extraer_tasas_sbs(
        html,
        fecha,
        moneda
    )

    # --------------------------------------------------------
    # FILTRAR GRUPO
    # --------------------------------------------------------

    if grupo is not None:

        df = df[
            df["grupo_credito"] == grupo
        ].copy()

        if df.empty:
            raise ValueError(
                f"No se encontró el grupo: {grupo}"
            )

    return df

In [32]:
df_corporativos = obtener_tasas_sbs(
    "15/09/2026",
    "MN",
    "Corporativos"
)

display(df_corporativos)

,fecha,moneda,tipo_credito,grupo_credito,BBVA,Bancom,Crédito,Pichincha,BIF,Scotiabank,...,Santander,Ripley,Alfin,ICBC,Bank of China,BCI,Compartamos,Santander Cons. Bank,Efectiva,Promedio
0,15/09/2026,MN,Corporativos,Corporativos,4.84,6.66,5.10,4.76,6.01,5.34,...,7.35,NaN,11.18,5.63,NaN,5.35,NaN,NaN,NaN,5.15
1,15/09/2026,MN,Descuentos,Corporativos,5.82,NaN,6.18,NaN,6.35,6.16,...,8.38,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.16
2,15/09/2026,MN,Préstamos hasta 30 días,Corporativos,4.75,NaN,4.56,4.65,NaN,4.49,...,7.11,NaN,7.08,NaN,NaN,NaN,NaN,NaN,NaN,4.72
3,15/09/2026,MN,Préstamos de 31 a 90 días,Corporativos,5.00,7.05,5.03,4.62,4.81,5.87,...,7.59,NaN,13.00,5.38,NaN,5.35,NaN,NaN,NaN,5.34
4,15/09/2026,MN,Préstamos de 91 a 180 días,Corporativos,6.31,4.65,5.76,4.97,6.42,6.19,...,7.89,NaN,NaN,6.37,NaN,5.34,NaN,NaN,NaN,6.02
5,15/09/2026,MN,Préstamos de 181 a 360 días,Corporativos,5.48,NaN,5.29,NaN,NaN,4.45,...,6.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.39
6,15/09/2026,MN,Préstamos a más de 360 días,Corporativos,4.88,NaN,5.07,4.67,4.89,4.81,...,NaN,NaN,NaN,4.43,NaN,NaN,NaN,NaN,NaN,5.06


In [34]:
df_medianas = obtener_tasas_sbs(
    "15/09/2026",
    "ME",
    "Medianas Empresas"
)

display(df_medianas)

,fecha,moneda,tipo_credito,grupo_credito,BBVA,Bancom,Crédito,Pichincha,BIF,Scotiabank,...,Santander,Ripley,Alfin,ICBC,Bank of China,BCI,Compartamos,Santander Cons. Bank,Efectiva,Promedio
14,15/09/2026,ME,Medianas Empresas,Medianas Empresas,8.88,9.52,7.53,8.47,8.32,7.52,...,8.49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.87
15,15/09/2026,ME,Descuentos,Medianas Empresas,8.93,NaN,8.40,10.52,13.59,8.06,...,9.37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.57
16,15/09/2026,ME,Préstamos hasta 30 días,Medianas Empresas,8.08,NaN,7.23,NaN,NaN,7.67,...,5.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.31
17,15/09/2026,ME,Préstamos de 31 a 90 días,Medianas Empresas,8.92,10.31,5.71,10.50,7.34,8.21,...,6.27,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.39
18,15/09/2026,ME,Préstamos de 91 a 180 días,Medianas Empresas,9.49,7.95,8.80,6.55,9.14,7.65,...,8.61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.74
19,15/09/2026,ME,Préstamos de 181 a 360 días,Medianas Empresas,10.00,12.00,8.20,11.32,7.00,6.45,...,10.29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.83
20,15/09/2026,ME,Préstamos a más de 360 días,Medianas Empresas,7.76,NaN,8.12,8.15,9.50,6.91,...,10.29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.98


In [36]:
df_medianas = obtener_tasas_sbs(
    "15/09/2026",
    "MN",
    "Microempresas"
)

display(df_medianas)

,fecha,moneda,tipo_credito,grupo_credito,BBVA,Bancom,Crédito,Pichincha,BIF,Scotiabank,...,Santander,Ripley,Alfin,ICBC,Bank of China,BCI,Compartamos,Santander Cons. Bank,Efectiva,Promedio
28,15/09/2026,MN,Microempresas,Microempresas,21.90,21.66,64.58,36.43,22.57,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,76.45,NaN,42.72,66.87
29,15/09/2026,MN,Tarjetas de Crédito,Microempresas,34.22,NaN,52.45,36.43,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,41.49
30,15/09/2026,MN,Descuentos,Microempresas,9.40,NaN,27.85,NaN,22.57,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.50
31,15/09/2026,MN,Préstamos Revolventes,Microempresas,12.92,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,37.85,NaN,NaN,52.52
32,15/09/2026,MN,Préstamos a cuota fija hasta 30 días,Microempresas,35.33,NaN,94.44,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,64.40,NaN,NaN,71.68
33,15/09/2026,MN,Préstamos a cuota fija de 31 a 90 días,Microempresas,17.27,NaN,95.86,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,110.64,NaN,NaN,79.60
34,15/09/2026,MN,Préstamos a cuota fija de 91 a 180 días,Microempresas,24.79,NaN,74.48,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,111.31,NaN,76.70,107.81
35,15/09/2026,MN,Préstamos a cuota fija de 181 a 360 días,Microempresas,22.32,NaN,69.73,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,56.41,NaN,43.35,61.09
36,15/09/2026,MN,Préstamos a cuota fija a más de 360 días,Microempresas,19.39,21.66,58.95,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,35.31,NaN,42.68,40.42
